In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pasta_selic = BASE_DIR / "data" / "bronze" / "selic"

arquivos = list(pasta_selic.glob("*.json"))
assert len(arquivos) > 0, "Nenhum arquivo encontrado em data/bronze/selic/"

In [32]:
df_raw = pd.read_json(arquivos[0])
print(f"Arquivo: {arquivos[0].name} | Linhas: {len(df_raw)}")
df_raw.head()

Arquivo: selic_2026-09-06.json | Linhas: 251


,data,valor
0,08/09/2025,0.055131
1,09/09/2025,0.055131
2,10/09/2025,0.055131
3,11/09/2025,0.055131
4,12/09/2025,0.055131


In [33]:
df = df_raw.copy()
df = df.rename(columns={"valor": "taxa_selic_diaria"})
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["taxa_selic_diaria"] = pd.to_numeric(df["taxa_selic_diaria"], errors="coerce")
df = df.sort_values("data").reset_index(drop=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   data               251 non-null    datetime64[us]
 1   taxa_selic_diaria  251 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 4.1 KB


In [34]:
assert not df["data"].duplicated().any(), "Erro: Datas duplicadas na série Selic!"
assert df["taxa_selic_diaria"].isnull().sum() == 0, "Erro: Valores nulos encontrados na Selic!"
print("Auditoria de chaves e nulos: APROVADA.")

Auditoria de chaves e nulos: APROVADA.


In [35]:
display(df["taxa_selic_diaria"].describe())
assert (df["taxa_selic_diaria"] >= 0).all(), "Erro: Taxa Selic diária com valor negativo!"

count    251.000000
mean       0.054140
std        0.001225
min        0.051660
25%        0.053400
50%        0.055131
75%        0.055131
max        0.055131
Name: taxa_selic_diaria, dtype: float64

In [36]:
# 1. Define o diretório de destino na camada silver
pasta_silver = BASE_DIR / "data" / "silver"

# 2. Cria a pasta fisicamente no disco caso ela não exista
pasta_silver.mkdir(parents=True, exist_ok=True)

# 3. Define o caminho final do arquivo Parquet
caminho_teste_parquet = pasta_silver / "selic_silver_test.parquet"

# 4. Grava o DataFrame em formato Parquet
df.to_parquet(caminho_teste_parquet, index=False)

print(f"Teste de escrita Parquet com schema tipado concluído com sucesso!")
print(f"Arquivo salvo em: {caminho_teste_parquet.relative_to(BASE_DIR)}")

Teste de escrita Parquet com schema tipado concluído com sucesso!
Arquivo salvo em: data/silver/selic_silver_test.parquet
